In [13]:
import os, time, pandas as pd
from dataclasses import dataclass, asdict
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# -------------------- config --------------------
CNEL_URL = "https://www.cnel.it/Archivio-Contratti-Collettivi/Archivio-Nazionale-dei-contratti-e-degli-accordi-collettivi-di-lavoro/Contrattazione-Nazionale/Ricerca-CCNL"

def make_driver():
    opts = Options()
    opts.add_argument("--no-sandbox")
    opts.add_argument("--window-size=1920,1080")
    # Anti-bot bypass helps government firewalls stay calm
    opts.add_argument("--disable-blink-features=AutomationControlled")
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=opts)

@dataclass
class AgreementItaly:
    titolo: str; tipologia: str; codice_cnel: str; data_stipula: str; pdf_link: str | None

def scrape_italy_golden_combo(out_dir="italy_data", target_count=50):
    abs_out_dir = os.path.abspath(out_dir)
    os.makedirs(abs_out_dir, exist_ok=True)
    
    d = make_driver()
    results = []

    try:
        print(f"Loading {CNEL_URL}...")
        d.get(CNEL_URL)
        time.sleep(2)
        
        # 1. USE THE WORKING CERCA LOGIC
        print("Attempting to trigger search automatically...")
        try:
            # BROAD CSS SELECTOR - This is what worked for you
            search_btn = d.find_element(By.CSS_SELECTOR, "input[type='submit'], .btn-primary, button[id*='submit']")
            d.execute_script("arguments[0].click();", search_btn)
            print("   ---> 'Cerca' button clicked.")
        except:
            print("Auto-click failed. PLEASE CLICK 'CERCA' MANUALLY IN THE BROWSER NOW.")

        # 2. USE THE WORKING MAGNET LOGIC
        print("Waiting for results (Looking for 'Scarica allegato' icons)...")
        # Timeout set to 60s to allow for manual click backup
        WebDriverWait(d, 60).until(
            EC.presence_of_element_located((By.XPATH, "//a[@title='Scarica allegato']"))
        )
        
        print("Table detected! Scraping now...")
        time.sleep(3) # Give it a second to fully stabilize

        # Find all download links (these are our "anchors" for each row)
        download_links = d.find_elements(By.XPATH, "//a[@title='Scarica allegato']")
        
        for i, link in enumerate(download_links[:target_count]):
            try:
                # Find the row (tr) that contains this specific link
                row = link.find_element(By.XPATH, "./ancestor::tr")
                cols = row.find_elements(By.TAG_NAME, "td")
                
                if len(cols) < 5: continue
                
                # Mapping from your 8-column screenshot
                # [1] Titolo | [2] Tipologia | [3] Codice | [4] Data Stipula
                titolo = cols[1].text.strip()
                tipologia = cols[2].text.strip()
                codice = cols[3].text.strip()
                data = cols[4].text.strip()
                pdf_url = link.get_attribute("href")

                results.append(AgreementItaly(
                    titolo=titolo, tipologia=tipologia, 
                    codice_cnel=codice, data_stipula=data, 
                    pdf_link=pdf_url
                ))
                
                if (i+1) % 5 == 0:
                    print(f"   [{i+1}/{target_count}] Captured: {codice}")
                    
            except Exception as e:
                print(f"   [!] Error on row {i}: {e}")

    finally:
        d.quit()
        if results:
            df = pd.DataFrame([asdict(r) for r in results])
            csv_path = os.path.join(abs_out_dir, "italy_final_links.csv")
            # utf-8-sig keeps the Italian accents clean for Excel
            df.to_csv(csv_path, index=False, encoding="utf-8-sig")
            print(f"\nSUCCESS! {len(results)} links saved to {csv_path}")
            return df

# RUN
scrape_italy_golden_combo(target_count=50)

Loading https://www.cnel.it/Archivio-Contratti-Collettivi/Archivio-Nazionale-dei-contratti-e-degli-accordi-collettivi-di-lavoro/Contrattazione-Nazionale/Ricerca-CCNL...
Attempting to trigger search automatically...
   ---> 'Cerca' button clicked.
Waiting for results (Looking for 'Scarica allegato' icons)...
Table detected! Scraping now...
   [5/50] Captured: S105
   [10/50] Captured: I241
   [15/50] Captured: C058
   [20/50] Captured: K51F

SUCCESS! 20 links saved to /Users/lukeschreiber/Downloads/RA Work/Collective-Bargaining-Agreements-Web-Scraping/italy_scraping/italy_data/italy_final_links.csv


,titolo,tipologia,codice_cnel,data_stipula,pdf_link
0,Accordo integrativo al CCNL per i dipendenti d...,Verbale integrativo,C05A,04/03/2026,https://static.cnel.it/archivio/2026/73feb1e0-...
1,CCNL del personale dell'area Presidenza del Co...,Testo definitivo,S003,27/02/2026,https://static.cnel.it/archivio/2026/ef15e874-...
2,CCNL dell'Area Sanità - triennio 2022-2024,Testo definitivo,S225,27/02/2026,https://static.cnel.it/archivio/2026/63cb7e1d-...
3,Disposizioni specifiche per gli addetti all'in...,Verbale integrativo,E012,26/02/2026,https://static.cnel.it/archivio/2026/e35be9f1-...
4,CCNL relativo al personale del Comparto Funzio...,Testo definitivo,S105,23/02/2026,https://static.cnel.it/archivio/2026/6fcd38a4-...
5,CCNL relativo al personale dell'area delle Fun...,Testo definitivo,S125,23/02/2026,https://static.cnel.it/archivio/2026/b43985e6-...
6,Accordo integrativo al CCNL sulla disciplina d...,Accordo economico,H52D,16/02/2026,https://static.cnel.it/archivio/2026/fbe6c694-...
7,"CCNL Estetica, Parrucchieri, Acconciatura - Te...",Testo definitivo,H52D,16/02/2026,https://static.cnel.it/archivio/2026/de6bb1a2-...
8,CCNL per i lavoratori delle imprese radiofonic...,Testo definitivo,G099,16/02/2026,https://static.cnel.it/archivio/2026/a4c971fe-...
9,Accordo di rinnovo del CCNL 31 maggio 2023 per...,Accordo di rinnovo,I241,16/02/2026,https://static.cnel.it/archivio/2026/18e881b0-...
